# Backgrounds Challenge — Fine-tuning ResNet-50 avec MIXED-RAND simulé

Stratégie : générer des masques foreground sur `in9/train/` avec DeepLabV3, puis entraîner sur des images composite (foreground classe A + fond aléatoire classe B≠A), ce qui simule le dataset MIXED-RAND du papier sans télécharger les données originales.

In [ ]:
# Cell 1 — Imports & device
import os
import json
import time
import random
import shutil
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, datasets
from torchvision.models import resnet50, ResNet50_Weights
from torchvision.models.segmentation import deeplabv3_resnet50, DeepLabV3_ResNet50_Weights

device = "cuda:0" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
print(f"{torch.__version__=}")
print(f"Using {device=}")

In [ ]:
# Cell 2 — Mount Google Drive + copie locale
from google.colab import drive
drive.mount('/content/drive')

DRIVE_PATH = "/content/drive/MyDrive/in9"
TRAIN_PATH = "/content/in9"

if not os.path.exists(TRAIN_PATH):
    print("Copie de in9/ depuis Drive...")
    t0 = time.time()
    shutil.copytree(DRIVE_PATH, TRAIN_PATH)
    print(f"Copie terminée en {time.time()-t0:.0f}s")
else:
    print("in9/ déjà présent localement.")

print(f"Train data path: {TRAIN_PATH}")

In [ ]:
# Cell 3 — Download test data (bg_challenge)
!wget -q "https://github.com/MadryLab/backgrounds_challenge/releases/download/data/backgrounds_challenge_data.tar.gz"
!tar -xzf backgrounds_challenge_data.tar.gz
!ls bg_challenge/

In [ ]:
# Cell 4 — Génération des masques foreground avec DeepLabV3
#
# DeepLabV3 est entraîné sur PASCAL VOC (21 classes).
# Classe 0 = background → tout pixel prédit != 0 est considéré foreground.
# Les masques sont sauvegardés en .npy (uint8, 0/1) à côté des images.
#
# Temps estimé sur T4 : ~5-8 minutes pour 12k images (batch=32 sur GPU)

MASK_DIR = "/content/in9_masks"

if os.path.exists(MASK_DIR) and len(os.listdir(MASK_DIR)) == 9:
    print("Masques déjà générés, étape ignorée.")
else:
    print("Chargement de DeepLabV3...")
    seg_model = deeplabv3_resnet50(weights=DeepLabV3_ResNet50_Weights.DEFAULT).to(device)
    seg_model.eval()

    seg_transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])

    os.makedirs(MASK_DIR, exist_ok=True)
    train_img_dir = f"{TRAIN_PATH}/train"
    BATCH_SIZE_SEG = 32
    total_imgs = 0
    t0 = time.time()

    for class_dir in sorted(os.listdir(train_img_dir)):
        class_path = os.path.join(train_img_dir, class_dir)
        mask_class_path = os.path.join(MASK_DIR, class_dir)
        os.makedirs(mask_class_path, exist_ok=True)

        img_names = [f for f in os.listdir(class_path) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]

        for i in range(0, len(img_names), BATCH_SIZE_SEG):
            batch_names = img_names[i:i + BATCH_SIZE_SEG]
            batch_tensors = []

            for img_name in batch_names:
                img = Image.open(os.path.join(class_path, img_name)).convert('RGB')
                batch_tensors.append(seg_transform(img))

            batch = torch.stack(batch_tensors).to(device)
            with torch.no_grad():
                output = seg_model(batch)['out']  # (B, 21, H, W)

            # Foreground = tout pixel non-background (classe 0 dans PASCAL VOC)
            masks = (output.argmax(1) != 0).cpu().numpy().astype(np.uint8)  # (B, 224, 224)

            for j, img_name in enumerate(batch_names):
                mask = masks[j]
                # Fallback : si le modèle ne détecte rien, on considère tout l'image comme foreground
                if mask.sum() == 0:
                    mask = np.ones((224, 224), dtype=np.uint8)
                mask_name = os.path.splitext(img_name)[0] + '.npy'
                np.save(os.path.join(mask_class_path, mask_name), mask)

            total_imgs += len(batch_names)

        print(f"  Classe {class_dir}: {len(img_names)} masques générés")

    print(f"\nTotal : {total_imgs} masques en {time.time()-t0:.0f}s")

    # Libérer la mémoire GPU
    del seg_model
    torch.cuda.empty_cache()

In [ ]:
# Cell 5 — Vérification visuelle des masques (3 exemples)
fig, axes = plt.subplots(3, 3, figsize=(10, 10))
train_img_dir = f"{TRAIN_PATH}/train"

for row, class_dir in enumerate(['0', '1', '2']):
    class_path = os.path.join(train_img_dir, class_dir)
    img_name = os.listdir(class_path)[0]
    img = Image.open(os.path.join(class_path, img_name)).convert('RGB').resize((224, 224))
    mask_path = os.path.join(MASK_DIR, class_dir, os.path.splitext(img_name)[0] + '.npy')
    mask = np.load(mask_path)

    img_arr = np.array(img)
    fg_only = img_arr * mask[:, :, None]

    axes[row, 0].imshow(img); axes[row, 0].set_title(f'Classe {class_dir} — original')
    axes[row, 1].imshow(mask, cmap='gray'); axes[row, 1].set_title('Masque DeepLabV3')
    axes[row, 2].imshow(fg_only); axes[row, 2].set_title('Foreground isolé')

for ax in axes.flat:
    ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# Cell 6 — Dataset MIXED-RAND simulé
#
# Pour chaque image :
#   1. Charger l'image + son masque foreground
#   2. Tirer aléatoirement une image de fond d'une classe DIFFÉRENTE
#   3. Composite : fg_mask * img + (1-fg_mask) * bg_img
#   4. Appliquer les augmentations standard
#
# Résultat : le modèle voit toujours le bon foreground mais avec un fond
# d'une autre classe → impossible de tricher sur le fond.

class MixedRandDataset(Dataset):
    def __init__(self, img_dir, mask_dir, transform=None):
        self.base_dataset = datasets.ImageFolder(img_dir)
        self.mask_dir = mask_dir
        self.transform = transform

        # Index des images par classe pour le tirage aléatoire du fond
        self.class_to_indices = defaultdict(list)
        for idx, (_, label) in enumerate(self.base_dataset.samples):
            self.class_to_indices[label].append(idx)
        self.n_classes = len(self.class_to_indices)

    def __len__(self):
        return len(self.base_dataset)

    def _load_resized(self, path):
        return Image.open(path).convert('RGB').resize((224, 224), Image.BILINEAR)

    def __getitem__(self, idx):
        img_path, label = self.base_dataset.samples[idx]
        img = self._load_resized(img_path)

        # Charger le masque foreground
        class_dir = str(label)
        img_stem = os.path.splitext(os.path.basename(img_path))[0]
        mask_path = os.path.join(self.mask_dir, class_dir, img_stem + '.npy')
        fg_mask = np.load(mask_path).astype(np.float32)  # (224, 224), valeurs 0/1

        # Tirer un fond depuis une classe différente
        other_classes = [c for c in self.class_to_indices if c != label]
        bg_class = random.choice(other_classes)
        bg_idx = random.choice(self.class_to_indices[bg_class])
        bg_path, _ = self.base_dataset.samples[bg_idx]
        bg_img = self._load_resized(bg_path)

        # Composite pixel-wise
        fg_mask_3ch = fg_mask[:, :, None]                      # (224, 224, 1)
        img_arr = np.array(img, dtype=np.float32)
        bg_arr  = np.array(bg_img, dtype=np.float32)
        mixed   = fg_mask_3ch * img_arr + (1 - fg_mask_3ch) * bg_arr
        mixed_img = Image.fromarray(mixed.clip(0, 255).astype(np.uint8))

        if self.transform:
            mixed_img = self.transform(mixed_img)

        return mixed_img, label

print("MixedRandDataset défini.")

In [ ]:
# Cell 7 — Transforms & DataLoaders
train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])
# Pas de RandomResizedCrop ici : le composite est déjà à 224x224
# et le recadrage aléatoire couperait le foreground composite

val_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

train_ds = MixedRandDataset(f"{TRAIN_PATH}/train", MASK_DIR, transform=train_transform)
val_ds   = datasets.ImageFolder(f"{TRAIN_PATH}/val", transform=val_transform)

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=64, shuffle=False, num_workers=2, pin_memory=True)

print(f"Train: {len(train_ds)} images, {len(train_loader)} batches")
print(f"Val:   {len(val_ds)} images, {len(val_loader)} batches")

In [ ]:
# Cell 8 — Vérification visuelle du dataset composite (4 exemples)
denorm = transforms.Normalize(
    mean=[-0.485/0.229, -0.456/0.224, -0.406/0.225],
    std=[1/0.229, 1/0.224, 1/0.225]
)
IN9_CLASSES = ['Dog', 'Bird', 'Vehicle', 'Reptile', 'Carnivore', 'Insect', 'Instrument', 'Primate', 'Fish']

fig, axes = plt.subplots(1, 4, figsize=(14, 4))
for i in range(4):
    img_t, label = train_ds[i * 1000]
    img_show = denorm(img_t).permute(1, 2, 0).numpy().clip(0, 1)
    axes[i].imshow(img_show)
    axes[i].set_title(f'{IN9_CLASSES[label]}\n(fond aléatoire)')
    axes[i].axis('off')
plt.suptitle('Exemples MIXED-RAND simulé')
plt.tight_layout()
plt.show()

In [ ]:
# Cell 9 — Modèle ResNet-50 (9 classes)
NUM_CLASSES = 9
model = resnet50(weights=ResNet50_Weights.IMAGENET1K_V1)
model.fc = nn.Linear(model.fc.in_features, NUM_CLASSES)
model = model.to(device)
print(f"Model: ResNet-50, fc: {model.fc}")

In [ ]:
# Cell 10 — Optimizer, scheduler, fonctions train/val
EPOCHS = 10

optimizer = torch.optim.SGD(model.parameters(), lr=0.01, momentum=0.9, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
criterion = nn.CrossEntropyLoss()

def train_epoch(model, loader):
    model.train()
    total_loss, n_batches = 0.0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        loss = criterion(model(images), labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        n_batches += 1
    return total_loss / n_batches

def eval_epoch(model, loader):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            preds = model(images).argmax(1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    return correct / total

print(f"Training setup: {EPOCHS} epochs, lr=0.01, SGD + CosineAnnealing")

In [ ]:
# Cell 11 — Boucle d'entraînement
history = {'train_loss': [], 'val_acc': []}

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    train_loss = train_epoch(model, train_loader)
    val_acc    = eval_epoch(model, val_loader)
    scheduler.step()
    elapsed = time.time() - t0

    history['train_loss'].append(train_loss)
    history['val_acc'].append(val_acc)
    print(f"Epoch {epoch:2d}/{EPOCHS} | loss: {train_loss:.4f} | val_acc: {val_acc*100:.2f}% | {elapsed:.0f}s")

CKPT_PATH = "/content/drive/MyDrive/resnet50_mixedrand_in9.pth"
torch.save(model.state_dict(), CKPT_PATH)
print(f"\nCheckpoint saved to {CKPT_PATH}")

In [ ]:
# Cell 12 — Évaluation sur bg_challenge
def evaluate(variation, data_root="bg_challenge", batch_size=64):
    dataset = datasets.ImageFolder(f"{data_root}/{variation}/val", transform=val_transform)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=2)
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for images, targets in loader:
            images = images.to(device)
            preds = model(images).argmax(1).cpu()
            correct += (preds == targets).sum().item()
            total += targets.size(0)
    return correct / total

variations = ["original", "mixed_same", "mixed_rand", "only_fg", "no_fg", "only_bg_t", "only_bg_b"]
results_mr = {}

for var in variations:
    t0 = time.time()
    acc = evaluate(var)
    results_mr[var] = acc
    print(f"{var:15s}: {acc*100:.1f}%  ({time.time()-t0:.0f}s)")

bg_gap_mr = (results_mr['mixed_same'] - results_mr['mixed_rand']) * 100
print(f"\nBG-Gap (MIXED-RAND simulé): {bg_gap_mr:.1f}%")

In [ ]:
# Cell 13 — Comparaison baseline / Mixup / MIXED-RAND simulé
results_baseline = {
    'original': 0.955, 'mixed_same': 0.862, 'mixed_rand': 0.789,
    'only_fg': 0.866, 'no_fg': 0.462, 'only_bg_t': 0.159, 'only_bg_b': 0.116,
}
results_mixup = {
    'original': 0.959, 'mixed_same': 0.916, 'mixed_rand': 0.804,
    'only_fg': 0.900, 'no_fg': 0.526, 'only_bg_t': 0.418, 'only_bg_b': 0.299,
}
bg_gap_baseline = (results_baseline['mixed_same'] - results_baseline['mixed_rand']) * 100
bg_gap_mixup    = (results_mixup['mixed_same']    - results_mixup['mixed_rand'])    * 100

print(f"{'Variation':15s} | {'Baseline':>9s} | {'Mixup':>9s} | {'MixedRand':>9s}")
print("-" * 55)
for var in variations:
    b = results_baseline[var] * 100
    m = results_mixup[var] * 100
    r = results_mr[var] * 100
    print(f"{var:15s} | {b:>8.1f}% | {m:>8.1f}% | {r:>8.1f}%")
print("-" * 55)
print(f"{'BG-Gap':15s} | {bg_gap_baseline:>8.1f}% | {bg_gap_mixup:>8.1f}% | {bg_gap_mr:>8.1f}%")

# Bar chart 3 modèles
x = np.arange(len(variations))
w = 0.25
fig, ax = plt.subplots(figsize=(13, 5))
b1 = ax.bar(x - w, [results_baseline[v]*100 for v in variations], w, label='Baseline', color='steelblue')
b2 = ax.bar(x,     [results_mixup[v]*100    for v in variations], w, label='Mixup α=0.2', color='coral')
b3 = ax.bar(x + w, [results_mr[v]*100       for v in variations], w, label='MIXED-RAND simulé', color='mediumseagreen')
ax.bar_label(b1, fmt='%.0f%%', padding=2, fontsize=7)
ax.bar_label(b2, fmt='%.0f%%', padding=2, fontsize=7)
ax.bar_label(b3, fmt='%.0f%%', padding=2, fontsize=7)
ax.set_xticks(x)
ax.set_xticklabels(variations, rotation=20, ha='right')
ax.set_ylim(0, 120)
ax.set_ylabel('Accuracy (%)')
ax.set_title(
    f'BG-Gap — Baseline: {bg_gap_baseline:.1f}%  |  Mixup: {bg_gap_mixup:.1f}%  |  MIXED-RAND: {bg_gap_mr:.1f}%'
)
ax.legend()
plt.tight_layout()
plt.savefig("/content/drive/MyDrive/comparison_3models.png", dpi=150)
plt.show()
print("Figure saved.")